In [1]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI


In [2]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-35-turbo"
deployment = "gpt-35-turbo"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=512,
        temperature=0,
        model=deployment
    )

In [3]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

In [4]:
dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/AddSubsampled_train.json')

In [5]:
CoT_prompt_original_direct = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/COT_direct.txt').read()

In [6]:
# Main logic
acc = 0
total = 0

output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/outputs/test1Output.txt'

with open(output_path, 'w') as fd:
    for d in tqdm(dev_data):
        print(acc, total)
        q = d['question']
        a = float(d['correct'][0])  # Ground truth answer

        prompt_q = (
            CoT_prompt_original_direct +
            '\nQ: ' + q + " Think step by step. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # Try to extract answer after 'the answer is'
        match = re.search(r'the answer is\s*([-\d\.]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_ans = match.group(1).rstrip('.')  # Remove trailing period
        else:
            extracted_ans = "N/A"

        fd.write(f'Q: {q}\nA_model:\n{ans_model}\nExtracted:\n{extracted_ans}\nA:\n{a}\n\n')

        try:
            if float(extracted_ans) == a:
                acc += 1
        except ValueError:
            pass  # If extraction failed or not a valid float

        total += 1

print(f"Accuracy: {acc} / {total} = {acc/total:.2%}")

  0%|          | 0/200 [00:00<?, ?it/s]

0 0


  0%|          | 1/200 [00:01<03:46,  1.14s/it]

1 1


  1%|          | 2/200 [00:02<03:33,  1.08s/it]

...

166 198


100%|█████████▉| 199/200 [03:14<00:00,  1.02it/s]

167 199


100%|██████████| 200/200 [03:15<00:00,  1.02it/s]

Accuracy: 168 / 200 = 84.00%


In [7]:
CoCT_prompt_original_direct = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/CoCT_direct.txt').read()

1 Prompt CoCT - ACC 85.5

In [8]:
acc = 0
total = 0

output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/outputs/test1Output.txt'

def clean_and_truncate(value_str):
    """Clean currency or comma signs and truncate to 3 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # Remove all non-numeric (except dot and minus)
    try:
        num = float(cleaned)
        truncated = int(num * 1000) / 1000  # Truncate to 3 decimal places
        return truncated
    except ValueError:
        return None

with open(output_path, 'w') as fd:
    for d in tqdm(dev_data):
        print(acc, total)
        q = d['question']
        a = float(d['correct'][0])  # Ground truth answer

        prompt_q = (
            CoCT_prompt_original_direct +
            '\nQ: ' + q +
            "\nA: Let's think step by step. Propose two different ways to solve this problem. Compare their reasoning, and explain which is better. Then solve the problem using the better method. "
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert math tutor who solves word problems using step-by-step reasoning. "
                    "You always consider multiple solution strategies, compare them, and explain why the chosen method is best. However, you always prefer more direct strategies. "
                    "You end every response with a clearly stated numeric, float, answer in the form: Answer: <value>."
                )
            },
            {
                "role": "user",
                "content": prompt_q
            }
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # Extract and clean answer
        match = re.search(r'answer:\s*([-\d\.\$,]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        fd.write(f'Q: {q}\nA_model:\n{ans_model}\nExtracted:\n{extracted}\nA:\n{a}\n\n')

        if extracted is not None and extracted == round(a, 3):
            acc += 1

        total += 1

print(f"Accuracy: {acc} / {total} = {acc/total:.2%}")

  0%|          | 0/200 [00:00<?, ?it/s]

0 0


  0%|          | 1/200 [00:01<06:01,  1.82s/it]

1 1


  1%|          | 2/200 [00:04<07:25,  2.25s/it]

...

154 198


100%|█████████▉| 199/200 [04:45<00:01,  1.45s/it]

155 199


100%|██████████| 200/200 [04:47<00:00,  1.44s/it]

Accuracy: 156 / 200 = 78.00%


2 Prompt CoCT 

In [106]:
acc = 0
total = 0
error_count = 0

output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/outputs/test3Output.txt'
error_log_path = output_path.replace('.txt', '_errors.txt')

def clean_and_truncate(value_str):
    """Clean currency or comma signs and truncate to 3 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        truncated = int(num * 1000) / 1000
        return truncated
    except ValueError:
        return None

for d in tqdm(dev_data):
    log_error = False
    print(acc, total)
    q = d['question']
    a = float(d['correct'][0])

    reasoning_prompt = (
        CoCT_prompt_original_direct +
        '\nQ: ' + q +
        "\nA: Propose two different ways to solve this problem. Compare their reasoning and identify which is better. "
        "List as 'Method 1: <...>' and 'Method 2: <...>'. At the end, always conclude with: ['PrefMethod: Method 1'] or ['PrefMethod: Method 2']."
    )

    messages1 = [
        {
            "role": "system",
            "content": (
                "You are an expert who analyzes and compares multiple solution methods to word problems. "
                "You always prefer the most efficient and logical method. Think step by step. "
                "Do not solve the question yet. Only provide your reasoning and preferred method."
            )
        },
        {"role": "user", "content": reasoning_prompt}
    ]

    response1 = completion_with_backoff(messages1)
    reasoning_analysis = response1.choices[0].message.content.strip()

    # === Extract Preferred Method Label ===
    pref_match = re.search(r'PrefMethod:\s*(Method\s*\d+)', reasoning_analysis, re.IGNORECASE)
    preferred_label = pref_match.group(1).strip().lower() if pref_match else None

    if not preferred_label:
        error_count += 1
        preferred_method = "[ERROR] PrefMethod not found."
        log_error = True
    else:
        # Extract lines
        method_lines = reasoning_analysis.strip().splitlines()
        method1_line = next((line for line in method_lines if line.lower().startswith("method 1:")), None)
        method2_line = next((line for line in method_lines if line.lower().startswith("method 2:")), None)

        if preferred_label == "method 1" and method1_line:
            preferred_method = method1_line
        elif preferred_label == "method 2" and method2_line:
            preferred_method = method2_line
        else:
            preferred_method = "[ERROR] Could not match preferred label to method line."
            log_error = True
            error_count += 1

    # === Build Answer Prompt ===
    answer_prompt = (
        f"The preferred method to solve the problem is:\n{preferred_method}\n\n"
        "Now, solve the problem using only this method. "
        "Show your steps and write the final numeric answer in the form: Answer: <value>"
    )

    messages2 = [
        {
            "role": "system",
            "content": (
                "You are an expert math tutor solving the problem using the preferred method. "
                "Follow clear math steps and provide a final answer in the form: Answer: <value>."
            )
        },
        {"role": "user", "content": answer_prompt}
    ]

    response2 = completion_with_backoff(messages2)
    ans_model = response2.choices[0].message.content.strip()

    match = re.search(r'Answer:\s*([-\d\.\$,]+)', ans_model, re.IGNORECASE)
    if match:
        extracted_raw = match.group(1).strip().rstrip('.')
        extracted = clean_and_truncate(extracted_raw)
    else:
        extracted = None
        error_count += 1
        log_error = True

    # === Output Logging ===
    log_block = (
        f'Q: {q}\n'
        f'REASONING:\n{reasoning_analysis}\n'
        f'PREF_METHOD_LABEL:\n{preferred_label}\n'
        f'PREF_METHOD_TEXT:\n{preferred_method}\n'
        f'PROMPT_INPUT:\n{answer_prompt}\n'
        f'ANSWER:\n{ans_model}\n'
        f'Extracted:\n{extracted}\n'
        f'Ground Truth:\n{a}\n\n'
    )

    # Always write to output file
    with open(output_path, 'a') as fd:
        fd.write(log_block)

    if log_error:
        with open(error_log_path, 'a') as fe:
            fe.write("⚠️ ERROR ENCOUNTERED\n")
            fe.write(log_block)

    print(log_block)

    if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
        acc += 1

    total += 1

# === Save Final Results ===
summary = (
    f"✅ Accuracy: {acc} / {total} = {acc / total:.2%}\n"
    f"❌ Errors logged: {error_count} → see: {error_log_path}\n"
)

with open(output_path, 'a') as fd:
    fd.write("\n=== FINAL RESULTS ===\n")
    fd.write(summary)

print(summary)

  0%|          | 0/200 [00:00<?, ?it/s]

0 0


  0%|          | 1/200 [00:01<05:31,  1.66s/it]

Q: Melanie had 10 quarters and 17 pennies in her bank . Her dad gave her 27 pennies and her mother gave her 19 pennies . How many pennies does Melanie have now ? 
REASONING:
Method 1: Add all the pennies Melanie had and received: 17 + 27 + 19 = 63.
Method 2: Group the pennies received from her dad and mom first: 27 + 19 = 46, then add to the pennies she had: 17 + 46 = 63.
['PrefMethod: Method 1']
PREF_METHOD_LABEL:
method 1
PREF_METHOD_TEXT:
Method 1: Add all the pennies Melanie had and received: 17 + 27 + 19 = 63.
PROMPT_INPUT:
The preferred method to solve the problem is:
Method 1: Add all the pennies Melanie had and received: 17 + 27 + 19 = 63.

Now, solve the problem using only this method. Show your steps and write the final numeric answer in the form: Answer: <value>
ANSWER:
Adding all the pennies Melanie had and received:
17 (pennies she had) + 27 (pennies she received from her mom) + 19 (pennies she received from her dad) = 63

Therefore, Melanie had a total of 63 pennies.
Answ

  1%|          | 2/200 [00:04<08:01,  2.43s/it]

...

Q: Joan grew 29 carrots and 14 watermelons . Jessica grew 11 carrots . How many carrots did they grow in all ? 
REASONING:
Method 1: Add the number of carrots Joan and Jessica grew directly: 29 + 11 = 40.
Method 2: Group the carrots grown by Joan and Jessica first, then add the watermelons grown by Joan: 29 + 11 = 40.
PrefMethod: Method 1
PREF_METHOD_LABEL:
method 1
PREF_METHOD_TEXT:
Method 1: Add the number of carrots Joan and Jessica grew directly: 29 + 11 = 40.
PROMPT_INPUT:
The preferred method to solve the problem is:
Method 1: Add the number of carrots Joan and Jessica grew directly: 29 + 11 = 40.

Now, solve the problem using only this method. Show your steps and write the final numeric answer in the form: Answer: <value>
ANSWER:
Adding the number of carrots Joan and Jessica grew directly:
29 + 11 = 40.

Answer: 40.
Extracted:
40.0
Ground Truth:
40.0


3 4


  2%|▎         | 5/200 [00:08<04:49,  1.48s/it]

Q: There are 115 pencils in the drawer . Sara placed 100 pencils in the drawer . How many pencils are now there in all ? 
REASONING:
Method 1: Add the number of pencils in the drawer to the number Sara placed: 115 + 100 = 215.
Method 2: Round 115 to 100, then adjust: 100 + 100 = 200, and add the difference to return to the actual value: 200 + 15 = 215.
PrefMethod: Method 1
PREF_METHOD_LABEL:
method 1
PREF_METHOD_TEXT:
Method 1: Add the number of pencils in the drawer to the number Sara placed: 115 + 100 = 215.
PROMPT_INPUT:
The preferred method to solve the problem is:
Method 1: Add the number of pencils in the drawer to the number Sara placed: 115 + 100 = 215.

Now, solve the problem using only this method. Show your steps and write the final numeric answer in the form: Answer: <value>
ANSWER:
Adding the number of pencils in the drawer to the number Sara placed:
115 + 100 = 215.

Answer: 215.
Extracted:
215.0
Ground Truth:
215.0


4 5


  2%|▎         | 5/200 [00:09<06:28,  1.99s/it]


KeyboardInterrupt: 

In [108]:
import re
import math
from tqdm import tqdm

acc = 0
total = 0
error_count = 0

output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/outputs/test4Output.txt'
error_log_path = output_path.replace('.txt', '_errors.txt')

def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return round(float(cleaned), 3)
    except ValueError:
        return None

for d in tqdm(dev_data):
    log_error = False
    print(acc, total)
    q = d['question']
    a = float(d['correct'][0])  # Ground truth answer

    # === Step 1: Get hypothesis only ===
    hypothesis_prompt = (
        CoCT_prompt_original_direct +
        '\nQ: ' + q +
        "\nA: Propose two different methods to solve this problem. Then write a short hypothesis detailing the steps in the better method, and why. "
        "Do not calculate or solve. Label the hypothesis clearly with 'Hypothesis: <text>'."
    )

    messages1 = [
        {
            "role": "system",
            "content": (
                "You are a reasoning assistant. Generate two solution methods, then hypothesize which method is better and why. "
                "Base your decision on simplicity, logical structure, or required operations. Do not solve the problem or include numbers in the hypothesis."
            )
        },
        {"role": "user", "content": hypothesis_prompt}
    ]

    response1 = completion_with_backoff(messages1)
    reasoning_text = response1.choices[0].message.content.strip()

    # === Extract hypothesis
    hypo_match = re.search(r'Hypothesis:\s*(.*)', reasoning_text, re.IGNORECASE)
    hypothesis = hypo_match.group(1).strip() if hypo_match else "[ERROR: No Hypothesis Found]"
    if "ERROR" in hypothesis:
        log_error = True
        error_count += 1

    # === Step 2: Solve using only the hypothesis as context
    answer_prompt = (
        f"Based on the hypothesis about the best solution approach:\n{hypothesis}\n\n"
        f"Now solve the problem step by step. Finish with: Answer: <value>"
    )

    messages2 = [
        {
            "role": "system",
            "content": (
                "You are an expert math tutor solving this word problem using the guidance of a hypothesis. "
                "Follow logical steps aligned with the hypothesis. End with: Answer: <value>"
            )
        },
        {"role": "user", "content": answer_prompt}
    ]

    response2 = completion_with_backoff(messages2)
    ans_model = response2.choices[0].message.content.strip()

    # === Extract numeric answer
    match = re.search(r'Answer:\s*([-\d\.\$,]+)', ans_model, re.IGNORECASE)
    extracted = clean_and_truncate(match.group(1)) if match else None
    if extracted is None:
        log_error = True
        error_count += 1

    # === Logging
    log_block = (
        f'Q: {q}\n'
        f'HYPOTHESIS:\n{hypothesis}\n'
        f'PROMPT_INPUT:\n{answer_prompt}\n'
        f'ANSWER_RAW:\n{ans_model}\n'
        f'EXTRACTED:\n{extracted}\n'
        f'GROUND_TRUTH:\n{a}\n\n'
    )

    print(log_block)
    break

    with open(output_path, 'a') as fd:
        fd.write(log_block)
    if log_error:
        with open(error_log_path, 'a') as fe:
            fe.write("⚠️ ERROR ENCOUNTERED\n" + log_block)

    if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
        acc += 1
    total += 1

# === Final Summary
summary = f"✅ Accuracy: {acc} / {total} = {acc / total:.2%}\n❌ Errors: {error_count} logged to {error_log_path}\n"
print(summary)
with open(output_path, 'a') as fd:
    fd.write("\n=== FINAL RESULTS ===\n" + summary)


  0%|          | 0/200 [00:00<?, ?it/s]

0 0


  0%|          | 0/200 [00:02<?, ?it/s]

Q: Melanie had 10 quarters and 17 pennies in her bank . Her dad gave her 27 pennies and her mother gave her 19 pennies . How many pennies does Melanie have now ? 
HYPOTHESIS:
Method 2 may be better as it breaks down the problem into smaller steps by grouping the pennies received from her dad and mom first before adding to the pennies she had. This approach may make it easier to keep track of the different sets of pennies and reduce the chances of errors in calculation.
PROMPT_INPUT:
Based on the hypothesis about the best solution approach:
Method 2 may be better as it breaks down the problem into smaller steps by grouping the pennies received from her dad and mom first before adding to the pennies she had. This approach may make it easier to keep track of the different sets of pennies and reduce the chances of errors in calculation.

Now solve the problem step by step. Finish with: Answer: <value>
ANSWER_RAW:
Step 1: Calculate the total number of pennies received from her dad and mom.


ZeroDivisionError: division by zero

In [124]:

CoCT_prompt_original_direct = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/Planing_direct.txt').read()

In [125]:
import re
import math
from tqdm import tqdm

acc = 0
total = 0
error_count = 0

output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/outputs/test4Output.txt'
error_log_path = output_path.replace('.txt', '_errors.txt')

def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return round(float(cleaned), 3)
    except ValueError:
        return None

for d in tqdm(dev_data):
    log_error = False
    print(acc, total)
    q = d['question']
    a = float(d['correct'][0])  # Ground truth answer

    # === Step 1: Planning prompt
    planning_prompt = (
        CoCT_prompt_original_direct +
        '\nQ: ' + q +
        "\nA: Propose two different methods to solve this problem. Then create a plan outlining what must be calculated or compared, and why one method is more efficient. "
        "Label this part as: Plan: <text>. Do not solve the problem or use numbers."
    )

    messages1 = [
        {
            "role": "system",
            "content": (
                "You are a math reasoning expert. Propose two logical methods to solve the problem, then create a clear plan that outlines the necessary steps to solve it. "
                "The plan should mention what needs to be calculated and why one method is better, but should NOT include numbers or an answer."
            )
        },
        {"role": "user", "content": planning_prompt}
    ]

    response1 = completion_with_backoff(messages1)
    reasoning_text = response1.choices[0].message.content.strip()

    # === Extract plan
    plan_match = re.search(r'Plan:\s*(.*)', reasoning_text, re.IGNORECASE)
    plan = plan_match.group(1).strip() if plan_match else "[ERROR: No Plan Found]"
    if "ERROR" in plan:
        log_error = True
        error_count += 1

    # === Step 2: Solve using plan
    answer_prompt = (
        f"Here is the question:\n{q}\n\n"
        f"Here is a plan for solving the problem:\n{plan}\n\n"
        f"Now, solve the problem step by step according to this plan. Finish with: Answer: <value, float numeric value only, convert fractions>"
    )

    messages2 = [
        {
            "role": "system",
            "content": (
                "You are an expert tutor solving a math word problem by following a structured plan. "
                "Do not change the plan. Stick to the logic and clearly show all steps. End with: Answer: <value>"
            )
        },
        {"role": "user", "content": answer_prompt}
    ]

    response2 = completion_with_backoff(messages2)
    ans_model = response2.choices[0].message.content.strip()

    # === Extract numeric answer
    match = re.search(r'Answer:\s*([-\d\.\$,]+)', ans_model, re.IGNORECASE)
    extracted = clean_and_truncate(match.group(1)) if match else None
    if extracted is None:
        log_error = True
        error_count += 1

    # === Log results
    log_block = (
        f'Q: {q}\n'
        f'PLAN:\n{plan}\n'
        f'PROMPT_INPUT:\n{answer_prompt}\n'
        f'ANSWER_RAW:\n{ans_model}\n'
        f'EXTRACTED:\n{extracted}\n'
        f'GROUND_TRUTH:\n{a}\n\n'
    )

    print(log_block)

    with open(output_path, 'a') as fd:
        fd.write(log_block)
    if log_error:
        with open(error_log_path, 'a') as fe:
            fe.write("⚠️ ERROR ENCOUNTERED\n" + log_block)

    if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
        acc += 1
    total += 1

# === Final Summary
summary = f"✅ Accuracy: {acc} / {total} = {acc / total:.2%}\n❌ Errors: {error_count} logged to {error_log_path}\n"
print(summary)
with open(output_path, 'a') as fd:
    fd.write("\n=== FINAL RESULTS ===\n" + summary)


  0%|          | 0/200 [00:00<?, ?it/s]

0 0


  0%|          | 1/200 [00:02<09:21,  2.82s/it]

Q: Melanie had 10 quarters and 17 pennies in her bank . Her dad gave her 27 pennies and her mother gave her 19 pennies . How many pennies does Melanie have now ? 
PLAN:
We need to determine the total number of pennies Melanie has now. The more efficient method is to use Method 1, as it involves a direct addition of all the pennies Melanie had initially and the pennies she received from her dad and mother. This method simplifies the calculation process and provides a straightforward solution to the problem.
PROMPT_INPUT:
Here is the question:
Melanie had 10 quarters and 17 pennies in her bank . Her dad gave her 27 pennies and her mother gave her 19 pennies . How many pennies does Melanie have now ? 

Here is a plan for solving the problem:
We need to determine the total number of pennies Melanie has now. The more efficient method is to use Method 1, as it involves a direct addition of all the pennies Melanie had initially and the pennies she received from her dad and mother. This method

  1%|          | 2/200 [00:06<10:23,  3.15s/it]

...

Q: In Shannon 's apartment complex , 0.16666666666666666 the apartments are one-bedroom apartments and 0.3333333333333333 are two-bedroom apartments . What fraction of the apartments are either 1 - or two-bedroom apartments ? 
PLAN:
To solve the problem, convert the decimals to fractions and add them together in Method 1. In Method 2, subtract the fraction of apartments that are not one- or two-bedroom from 1 to find the fraction of one- or two-bedroom apartments. Compare the two methods to determine which one is more efficient based on the complexity of calculations involved.
PROMPT_INPUT:
Here is the question:
In Shannon 's apartment complex , 0.16666666666666666 the apartments are one-bedroom apartments and 0.3333333333333333 are two-bedroom apartments . What fraction of the apartments are either 1 - or two-bedroom apartments ? 

Here is a plan for solving the problem:
To solve the problem, convert the decimals to fractions and add them together in Method 1. In Method 2, subtract th

100%|█████████▉| 199/200 [07:26<00:02,  2.21s/it]

Q: There were 73 bales of hay in the barn . Jason stacked bales in the barn today . There are now 96 bales of hay in the barn . How many bales did he store in the barn ? 
PLAN:
We need to determine how many bales of hay Jason stored in the barn. To find this, we can either subtract the initial number of bales from the current total to calculate the number of bales Jason added (Method 1), or we can add the number of bales Jason added to the initial amount to find the final total (Method 2). Method 1 may be more efficient as it directly calculates the number of bales added without needing to calculate the final total first.
PROMPT_INPUT:
Here is the question:
There were 73 bales of hay in the barn . Jason stacked bales in the barn today . There are now 96 bales of hay in the barn . How many bales did he store in the barn ? 

Here is a plan for solving the problem:
We need to determine how many bales of hay Jason stored in the barn. To find this, we can either subtract the initial number 

100%|██████████| 200/200 [07:29<00:00,  2.25s/it]

Q: Dina made cookies . She used 0.625 cup of flour and 0.25 cup of sugar . How much more flour than sugar did Dina use ? 
PLAN:
We need to determine how much more flour Dina used than sugar. To do this, we can convert the fractions to a common unit in Method 1 or find a common denominator in Method 2. Comparing the two methods, Method 1 may be more efficient as it involves converting to a common unit directly, simplifying the calculation process.
PROMPT_INPUT:
Here is the question:
Dina made cookies . She used 0.625 cup of flour and 0.25 cup of sugar . How much more flour than sugar did Dina use ? 

Here is a plan for solving the problem:
We need to determine how much more flour Dina used than sugar. To do this, we can convert the fractions to a common unit in Method 1 or find a common denominator in Method 2. Comparing the two methods, Method 1 may be more efficient as it involves converting to a common unit directly, simplifying the calculation process.

Now, solve the problem step b

In [65]:
print('Total %d correct %d acc %.4f' % (total, acc, acc / total))

Total 200 correct 162 acc 0.8100
